# Fine-tuning Gemma 4 to write issue tracker entries

Turns raw product input — a Slack message, a support ticket, a Sentry alert —
into a structured issue. Output is always one valid JSON object.

**Dataset:** [`fport/issue-writer-tr-en`](https://huggingface.co/datasets/fport/issue-writer-tr-en)
· 13k examples · 50% English, 50% Turkish
**Method:** QLoRA / LoRA via [Unsloth](https://unsloth.ai) · **Base:** `gemma-4-E4B-it`

### What you need

- A GPU. T4 (free Colab) works; L4 or A100 is 3–5× faster.
- A Hugging Face token, and the Gemma licence accepted once on the
  [model page](https://huggingface.co/google/gemma-4-E4B-it).
- About 2–3 hours on an A100, 4–6 on an L4.

### How it goes

1. Check the GPU and install Unsloth
2. Load the model and attach LoRA adapters
3. Prepare data and **verify the token lengths**
4. **Measure the base model** — without this you cannot claim an improvement
5. Mask the loss to assistant turns and **verify the mask**
6. Train
7. Measure again and compare
8. Save, and serve locally with Ollama or vLLM

Steps 3, 4 and 5 exist because each one caught a real failure during
development. The last section lists what went wrong and why the checks are
where they are.


---
## The stack, and which part does what

Two separate toolchains are involved and they are easy to confuse. **Training and
inference share almost nothing.**

### Training

| Library | Role |
|---|---|
| **transformers** | model and tokenizer definitions; everything else builds on it |
| **peft** | the LoRA implementation — the adapter matrices themselves |
| **trl** | `SFTTrainer`, the supervised fine-tuning loop |
| **bitsandbytes** | 4/8-bit quantisation and the 8-bit optimiser (`adamw_8bit`) |
| **unsloth** | custom Triton kernels that make the above ~2× faster on ~70% less VRAM |

Unsloth is **not** built on llama.cpp — a common misreading. It is a PyTorch/Triton
layer that patches attention, MLP and loss kernels. It only reaches for llama.cpp
when you ask it to export GGUF.

### Inference

| Tool | Role |
|---|---|
| **llama.cpp** | C++ inference engine and the GGUF format; no training, ever |
| **Ollama** | model management and a REST API on top of llama.cpp |
| **vLLM** | GPU serving with PagedAttention; takes safetensors, not GGUF |
| **llama-cpp-python** | Python bindings for llama.cpp — inference only |

So the path is: train with unsloth/peft/trl → save an adapter → either merge to
16-bit for vLLM, or convert to GGUF for Ollama. Nothing from the training stack
runs in production.

### What LoRA actually does

Full fine-tuning updates every weight — 8 billion of them here, which needs far more
memory than the model itself. LoRA freezes the original weights and inserts a pair of
small matrices next to each target layer. Only those train: **73M parameters, 0.9% of
the model.** The adapter is a few hundred MB instead of 16 GB, and you can keep
several of them for one base model.

QLoRA is the same idea with the frozen base held in 4-bit, which is what makes this
fit on a 16 GB card.


image.png

---
## 1 · GPU

The numbers here decide quantisation and batch size later, so run it first.


In [1]:
import torch, subprocess

print(subprocess.run(['nvidia-smi', '--query-gpu=name,memory.total',
                      '--format=csv,noheader'], capture_output=True, text=True).stdout)
assert torch.cuda.is_available(), 'No GPU. Runtime > Change runtime type > GPU'

VRAM = torch.cuda.get_device_properties(0).total_memory / 1024**3
BF16 = torch.cuda.is_bf16_supported()      # T4 is Turing: no bf16, fp16 only
print(f'{torch.cuda.get_device_name(0)} · {VRAM:.0f} GB · bf16 {BF16}')


NVIDIA A100-SXM4-80GB, 81920 MiB

NVIDIA A100-SXM4-80GB · 79 GB · bf16 True


## 2 · Install

If Colab offers to restart the session afterwards, you can decline and carry on.


In [2]:
%%capture
import os
if 'COLAB_' in ''.join(os.environ.keys()):
    !pip install --upgrade --no-cache-dir unsloth unsloth_zoo
    !pip install -q sentencepiece protobuf hf_transfer 'huggingface_hub>=0.34'
else:
    !pip install unsloth


In [3]:
import unsloth, transformers, trl
print('unsloth', unsloth.__version__, '| transformers', transformers.__version__,
      '| trl', trl.__version__)


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
unsloth 2026.9.2 | transformers 5.5.0 | trl 0.24.0


## 3 · Hugging Face token

Four sources are tried in order, and the cell prints which one it used:

1. `HF_TOKEN` environment variable
2. Colab *Secrets*
3. a saved `huggingface-cli login` session
4. interactive entry (hidden while typing)

> Colab Secrets only work **in the Colab web UI**. `userdata.get()` opens a
> permission dialog in the browser; from VS Code or a remote kernel there is no
> dialog and the call hangs until it times out. Hence the fallbacks.


In [4]:
import os


def get_hf_token() -> str:
    if tok := os.environ.get('HF_TOKEN'):
        print('token from: environment'); return tok
    try:
        from google.colab import userdata
        if tok := userdata.get('HF_TOKEN'):
            print('token from: Colab Secrets'); return tok
    except Exception as e:
        print(f'Colab Secrets unavailable ({type(e).__name__}), trying next')
    try:
        from huggingface_hub import get_token
        if tok := get_token():
            print('token from: saved huggingface-cli login'); return tok
    except Exception:
        pass
    from getpass import getpass
    return getpass('HF token (hidden): ').strip()


os.environ['HF_TOKEN'] = get_hf_token()
os.environ['HF_HUB_ENABLE_HF_TRANSFER'] = '1'

from huggingface_hub import whoami
print('signed in as', whoami()['name'])


Colab Secrets unavailable (TimeoutException), trying next
signed in as fport


## 4 · Load the model

**4-bit is a memory answer, not a speed one.** Above ~40 GB of VRAM it only adds
dequantisation cost to every matmul, so we load bf16 there and quantise only when
memory is actually short.

Batch sizes fill the card; the effective batch stays 16 either way, so the
learning dynamics do not change with the GPU.


In [5]:
from unsloth import FastModel

MODEL  = 'unsloth/gemma-4-E4B-it'      # ~4.5B effective, 8B total
MAXLEN = 2048
LOAD_4BIT = VRAM < 40

if VRAM >= 70:   BATCH, ACCUM = 8, 2       # A100 80GB
elif VRAM >= 35: BATCH, ACCUM = 4, 4       # A100 40GB
elif VRAM >= 22: BATCH, ACCUM = 2, 8       # L4 24GB
else:            BATCH, ACCUM = 1, 16      # T4 16GB

model, tokenizer = FastModel.from_pretrained(
    model_name      = MODEL,
    max_seq_length  = MAXLEN,
    load_in_4bit    = LOAD_4BIT,
    full_finetuning = False,
    token           = os.environ['HF_TOKEN'],
)
print(f"{model.config.model_type} | {'4-bit' if LOAD_4BIT else 'bf16'} "
      f"| effective batch {BATCH * ACCUM}")


==((====))==  Unsloth 2026.9.2: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

gemma4 | bf16 | effective batch 16


## 5 · Attach LoRA adapters

| Parameter | Value here | What it controls |
|---|---|---|
| `r` | 32 | Rank of the adapter matrices — the capacity of the fine-tune. Higher means more trainable parameters and a larger adapter. |
| `lora_alpha` | 64 | Scaling applied to the adapter output. The convention is `alpha = 2r`; what matters is the ratio, not the absolute number. |
| `lora_dropout` | 0 | Regularisation. Unsloth's fast path requires 0, and with 13k examples over 2 epochs overfitting is not the binding constraint. |
| `bias` | `none` | Whether bias terms train too. `none` is both faster and the usual choice. |
| `finetune_*_layers` | text only | Which parts of the model get adapters. Vision and audio encoders stay frozen — this is a text task, training them wastes compute. |
| `use_gradient_checkpointing` | `unsloth` | Recomputes activations instead of storing them: ~30% less VRAM for a little more compute. |
| `random_state` | 3407 | Reproducibility. |

**Why `r=32` and not the usual `r=8`.** Most LoRA guides fine-tune style or tone,
where 8 is plenty. Here the model has to learn a **fixed JSON schema** — field names,
required sections, the shape of an acceptance criterion. That is structure, and
structure needs capacity. If the schema metrics come out low after training, raising
`r` to 64 is a better first move than switching to a bigger model.

Expect roughly 0.9% trainable. Far lower means the target modules did not match; far
higher means the encoders are being trained too.


In [6]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,
    r            = 32,
    lora_alpha   = 64,
    lora_dropout = 0,
    bias         = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state = 3407,
)
model.print_trainable_parameters()


trainable params: 73,400,320 || all params: 8,069,556,768 || trainable%: 0.9096


## 6 · Data

The dataset ships `messages` (system / user / assistant). We render it through the
model's chat template into a `text` column.

> **Strip the leading `<bos>`.** The template emits one and `SFTTrainer` adds
> another during tokenisation. Two BOS tokens at the start of every sequence is a
> silent corruption — no error, just a worse model.


In [8]:
from unsloth.chat_templates import get_chat_template
from datasets import load_dataset

tokenizer = get_chat_template(tokenizer, chat_template='gemma-4')

ds = load_dataset('fport/issue-writer-tr-en')


def to_text(row):
    t = tokenizer.apply_chat_template(row['messages'], tokenize=False,
                                      add_generation_prompt=False)
    return {'text': t.removeprefix('<bos>')}


ds = ds.map(to_text, remove_columns=['messages', 'meta'], num_proc=2)
print(ds)
print('\n--- one rendered example ---')
print(ds['train'][0]['text'][:600])


data/train.jsonl: reconstructing file:   0%|          |  0.00B / 29.2MB            

data/train.jsonl: downloading bytes:           |  0.00B            

validation.jsonl:   0%|          | 0.00/2.93M [00:00<?, ?B/s]

test.jsonl:   0%|          | 0.00/2.95M [00:00<?, ?B/s]

Generating train split: 0 examples [00:00, ? examples/s]

...

Generating test split: 0 examples [00:00, ? examples/s]

Map (num_proc=2):   0%|          | 0/10895 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1052 [00:00<?, ? examples/s]

Map (num_proc=2):   0%|          | 0/1053 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 10895
    })
    validation: Dataset({
        features: ['text'],
        num_rows: 1052
    })
    test: Dataset({
        features: ['text'],
        num_rows: 1053
    })
})

--- one rendered example ---
<|turn>system
You are a Jira issue writer for a product engineering team. Convert the user's input into structured issue fields as JSON. Keep summaries under 80 characters and in the imperative mood. Do not add a type prefix to the summary. Put every unstated detail into `assumptions` or `clarifying_questions`.<turn|>
<|turn>user
Please write this up as a proper ticket.

---
Inside the story “Add recurring transfer to Transfers” we still need this one step done. Can you write the sub-task?
---<turn|>
<|turn>model
{
  "issue_type": "Sub-task",
  "summary": "Wire the feature flag and rollout con


### Verify lengths and BOS

Truncated examples produce JSON that never closes, and the model learns to emit
broken output. Check the rate before training, not after.

> Gemma 4 is multimodal, so what Unsloth hands back as `tokenizer` is a
> **Processor** whose first positional argument is `images`. Calling
> `tokenizer(text)` sends your text into the image slot. Use the inner text
> tokenizer for anything token-level.


In [9]:
import random

_tok = getattr(tokenizer, 'tokenizer', tokenizer)     # inner text tokenizer


def n_tokens(text: str) -> int:
    ids = _tok(text)['input_ids']
    return len(ids[0] if ids and isinstance(ids[0], list) else ids)


idx   = random.Random(0).sample(range(len(ds['train'])), 500)
lens  = sorted(n_tokens(t) for t in ds['train'].select(idx)['text'])
over  = sum(x > MAXLEN for x in lens) / len(lens)
print(f'median {lens[len(lens)//2]} · p95 {lens[int(len(lens)*.95)]} · max {lens[-1]}')
print(f'over MAXLEN={MAXLEN}: {over*100:.1f}%')
assert over < 0.05, 'too much truncation — raise MAXLEN'

ids = _tok(ds['train'][0]['text'])['input_ids']
ids = ids[0] if ids and isinstance(ids[0], list) else ids
n_bos = 0
for t in ids:
    if t == _tok.bos_token_id: n_bos += 1
    else: break
print(f'leading BOS tokens: {n_bos}')
assert n_bos <= 1, 'double BOS — removeprefix in to_text did not fire'
print('lengths and BOS look right')


median 570 · p95 1203 · max 1631
over MAXLEN=2048: 0.0%
leading BOS tokens: 0
lengths and BOS look right


## 7 · Inference helper

One function, used for the baseline, the sanity check and the final measurement.

Three things it gets right that are easy to get wrong:

- **The system prompt is copied verbatim from the training data.** Changing a word
  at inference moves the model off-distribution. Do not edit it.
- **The Processor cannot tokenise plain-string messages.** Its `tokenize=True`
  path expects multimodal content lists, so we render to text first and tokenise
  with the inner tokenizer — the same route the training data took.
- **`add_special_tokens=False`**, because the rendered template already carries
  `<bos>`. Same double-BOS trap as in step 6, on the inference side.


In [14]:
from transformers import TextStreamer

# Verbatim from the dataset. Do not reword.
SYSTEM_EN = ('You are a senior agile delivery assistant. You turn raw product input '
             'into well-formed Jira issues. Reply with a single valid JSON object and '
             'nothing else. Follow INVEST, write testable Given/When/Then acceptance '
             'criteria, and never invent facts: anything the input does not state goes '
             'into `assumptions` or `clarifying_questions`.')


def render(msgs) -> str:
    """Chat template to text. The Processor cannot tokenise plain-string
    messages, so we render first and tokenise with the inner tokenizer."""
    return tokenizer.apply_chat_template(msgs, add_generation_prompt=True,
                                         tokenize=False)


def ask(msgs, max_new=1400, stream=False):
    """One prompt. Use ask_batch for measurement."""
    enc = _tok(render(msgs), return_tensors='pt', add_special_tokens=False).to('cuda')
    kw = dict(max_new_tokens=max_new, do_sample=False,
              pad_token_id=_tok.pad_token_id or _tok.eos_token_id)
    if stream:
        kw['streamer'] = TextStreamer(_tok, skip_prompt=True)
    with torch.no_grad():
        out = model.generate(**enc, **kw)
    return _tok.decode(out[0][enc['input_ids'].shape[1]:],
                       skip_special_tokens=True).strip()


def ask_batch(batch_msgs, max_new=1400, bs=8):
    """Several prompts at once — 4-5x faster than one at a time.

    Padding must be on the LEFT for generation: with right padding the model
    continues from pad tokens and the output is garbage.
    """
    side = _tok.padding_side
    _tok.padding_side = 'left'
    if _tok.pad_token_id is None:
        _tok.pad_token = _tok.eos_token
    out_texts = []
    try:
        for i in range(0, len(batch_msgs), bs):
            chunk = [render(m) for m in batch_msgs[i:i + bs]]
            enc = _tok(chunk, return_tensors='pt', padding=True,
                       add_special_tokens=False).to('cuda')
            with torch.no_grad():
                out = model.generate(**enc, max_new_tokens=max_new, do_sample=False,
                                     pad_token_id=_tok.pad_token_id)
            n_in = enc['input_ids'].shape[1]      # left padding: same for every row
            out_texts += [_tok.decode(row[n_in:], skip_special_tokens=True).strip()
                          for row in out]
            print(f'  {min(i + bs, len(batch_msgs))}/{len(batch_msgs)}')
    finally:
        _tok.padding_side = side
    return out_texts


_probe = _tok(render([{'role': 'user', 'content': 'test'}]),
              add_special_tokens=False)['input_ids']
assert _probe.count(_tok.bos_token_id) <= 1, 'double BOS at inference'
print('inference helpers ready')

inference helpers ready


## 8 · Measure the base model

**Without a baseline, "it improved" is an impression, not a result.**

Base Gemma produces something that reads like a reasonable issue. Against this
dataset's contract it satisfies roughly one rule in nine: no `h2.` sections,
acceptance criteria with invented field names, `components` and `dor_check`
missing. Readable for a person, unusable for a pipeline. That gap is the job.

> `for_inference()` matters here. Unsloth prepares the model for **training** —
> gradient checkpointing on, KV cache off. Generating under those settings
> recomputes the whole sequence per token: measurement went from 8 minutes to
> 2 hours when this was missed. The function switches modes and switches back.


In [15]:
import json, re, collections
from unsloth import FastModel

_test = load_dataset('fport/issue-writer-tr-en', split='test')
_rows = [r for r in _test if r['meta']['task'] in ('draft_issue', 'bug_from_log')]

VER    = re.compile(r'\b\d+\.\d+(?:\.\d+)?\b')
PREFIX = re.compile(r'^\s*(\[(bug|story|task|epic)\]|(bug|story|task|epic)\s*[:\-])', re.I)
H2     = re.compile(r'^h2\. ', re.M)
REQ    = ('issue_type', 'summary', 'description', 'priority', 'labels',
          'components', 'dor_check')


def evaluate(n=25, label='model', bs=None):
    # Batch size follows the card. Generation holds a KV cache per row, so this
    # is the setting that decides whether measurement takes 5 minutes or 30.
    bs = bs or (16 if VRAM >= 70 else 8 if VRAM >= 35 else 4 if VRAM >= 22 else 2)
    print(f'batch size {bs}')
    FastModel.for_inference(model)          # KV cache on, checkpointing off
    agg  = collections.defaultdict(list)
    rows = _rows[:n]

    preds = ask_batch([r['messages'][:2] for r in rows], bs=bs)

    for raw, r in zip(preds, rows):
        gold = json.loads(r['messages'][2]['content'])

        # strict: bare JSON, what we want. lenient: JSON once a code fence is stripped.
        try:
            p = json.loads(raw); agg['json_strict'].append(1)
        except json.JSONDecodeError:
            agg['json_strict'].append(0)
            try:
                p = json.loads(raw[raw.find('{'):raw.rfind('}') + 1])
            except Exception:
                agg['json_lenient'].append(0); continue
        agg['json_lenient'].append(1)

        agg['has_all_fields'].append(int(all(k in p for k in REQ)))
        agg['type_acc'].append(int(p.get('issue_type') == gold.get('issue_type')))
        s = p.get('summary', '')
        agg['summary_ok'].append(int(0 < len(s) <= 120 and not PREFIX.match(s)))
        agg['sections_ok'].append(int(len(H2.findall(p.get('description', ''))) >= 3))

        acs = p.get('acceptance_criteria') or []
        if p.get('issue_type') == 'Story':
            agg['ac_count_ok'].append(int(3 <= len(acs) <= 7))
        if acs:                              # only score issues that carry criteria
            agg['ac_shape_ok'].append(int(all(
                all(k in a for k in ('id', 'given', 'when', 'then')) for a in acs)))

        # invented facts: versions in the output must appear in the input
        agg['no_hallucination'].append(int(
            set(VER.findall(json.dumps(p, ensure_ascii=False)))
            <= set(VER.findall(r['messages'][1]['content']))))

    model.gradient_checkpointing_enable()   # back to training mode
    model.config.use_cache = False
    model.train()

    print(f'\n=== {label} · {n} examples ===')
    for k in sorted(agg):
        v = agg[k]
        print(f'  {k:18} {sum(v)/len(v)*100:5.1f}%   ({sum(v)}/{len(v)})')
    return {k: sum(v) / len(v) for k, v in agg.items()}


BEFORE = evaluate(n=25, label='BASE MODEL')


batch size 16
  16/25
  25/25

=== BASE MODEL · 25 examples ===
  ac_count_ok         57.1%   (4/7)
  ac_shape_ok          0.0%   (0/9)
  has_all_fields       0.0%   (0/25)
  json_lenient       100.0%   (25/25)
  json_strict          0.0%   (0/25)
  no_hallucination   100.0%   (25/25)
  sections_ok          0.0%   (0/25)
  summary_ok          72.0%   (18/25)
  type_acc            28.0%   (7/25)


## 9 · Trainer

**Full epochs, not a step count.** A `max_steps=60` demo shows the model about 1%
of this dataset — enough to prove the pipeline runs, not enough to learn anything.

`lr=1e-4` with cosine decay; `2e-4` is fine for a short run but oscillates over a
long one. `eval_steps` lets you watch for overfitting: when eval loss stops falling
and turns up, everything after that is memorisation.

Two settings earn their place on a large GPU:

- **`group_by_length=True`.** Examples here run to ~640 tokens on average against a
  2048 limit, so fixed padding wastes **69% of every batch**. Grouping similar
  lengths together drops that to about 1% — roughly 3× more real tokens per second
  at no cost in quality.
- **Gradient checkpointing off when memory allows.** It trades compute for memory;
  with 40 GB+ free there is nothing to trade for, and turning it off buys another
  25–30%.

> The checkpoint directory is resolved **before** the trainer. `SFTConfig` fixes
> `output_dir` at construction, so mounting Drive afterwards has no effect and a
> run cannot resume.
### The settings that matter

| Parameter | Value | Why |
|---|---|---|
| `num_train_epochs` | 2 | Full passes over the data. `max_steps` is for smoke tests — 60 steps here would show the model 1% of it. |
| `per_device_train_batch_size` × `gradient_accumulation_steps` | = 16 | The effective batch. Accumulation lets a small card behave like a large one by delaying the weight update. |
| `learning_rate` | 1e-4 | LoRA tolerates rates ~10× higher than full fine-tuning. `2e-4` suits short runs; over two epochs it oscillates. |
| `lr_scheduler_type` | cosine | Decays smoothly to near zero, which settles the model at the end of training. |
| `warmup_ratio` | 0.03 | Ramps the rate up over the first 3% of steps so early batches do not jolt the weights. |
| `optim` | `adamw_8bit` | Adam keeps two state tensors per parameter; 8-bit quantises them and saves about 1.5 GB here. |
| `weight_decay` | 0.01 | Mild pull toward smaller weights. |
| `group_by_length` | True | Batches similar-length examples. Without it 69% of every batch was padding. |
| `eval_steps` | 200 | The overfitting alarm: when eval loss stops falling and turns up, stop. |
| `save_steps` | 200 | Resume points. A dropped Colab session costs 200 steps, not the whole run. |
| `seed` | 3407 | Reproducibility. |


In [16]:
from trl import SFTTrainer, SFTConfig

EPOCHS = 2.0

try:
    from google.colab import drive
    drive.mount('/content/drive')
    CKPT = '/content/drive/MyDrive/issue-writer-ckpt'
except Exception:
    CKPT = os.path.abspath('./checkpoints')
os.makedirs(CKPT, exist_ok=True)
print('checkpoints:', CKPT)

# Checkpointing recomputes activations to save memory. With plenty of VRAM that
# trade is pure loss, so switch it off there.
USE_CHECKPOINTING = VRAM < 40

trainer = SFTTrainer(
    model         = model,
    tokenizer     = tokenizer,
    train_dataset = ds['train'],
    eval_dataset  = ds['validation'].select(range(256)),
    args = SFTConfig(
        dataset_text_field          = 'text',
        max_seq_length              = MAXLEN,
        per_device_train_batch_size = BATCH,
        gradient_accumulation_steps = ACCUM,
        per_device_eval_batch_size  = BATCH,
        num_train_epochs  = EPOCHS,
        learning_rate     = 1e-4,
        lr_scheduler_type = 'cosine',
        warmup_ratio      = 0.03,
        optim             = 'adamw_8bit',
        weight_decay      = 0.01,
        group_by_length   = True,      # ~69% of each batch was padding without this
        gradient_checkpointing = USE_CHECKPOINTING,
        logging_steps     = 20,
        eval_strategy = 'steps', eval_steps = 200,
        save_strategy = 'steps', save_steps = 200, save_total_limit = 2,
        output_dir = CKPT,
        seed       = 3407,
        report_to  = 'none',
    ),
)
print(f"{len(ds['train'])} examples · {EPOCHS} epochs · "
      f"~{int(len(ds['train']) * EPOCHS / (BATCH * ACCUM))} steps")
print(f"gradient checkpointing: {'on' if USE_CHECKPOINTING else 'off (enough VRAM)'}")


Mounted at /content/drive


warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


checkpoints: /content/drive/MyDrive/issue-writer-ckpt
Unsloth: `group_by_length` is not supported by the installed transformers's SFTConfig and will be IGNORED - set `train_sampling_strategy = "group_by_length"` instead.


Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/10895 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=6):   0%|          | 0/256 [00:00<?, ? examples/s]

10895 examples · 2.0 epochs · ~1361 steps
gradient checkpointing: off (enough VRAM)


### Train on the answer only

Without this the model memorises your prompts along with the answers.

> TRL's `assistant_only_loss=True` needs a `{% generation %}` block in the chat
> template. Where the template lacks it, loss is computed over the whole sequence
> **silently**. Unsloth's `train_on_responses_only` takes explicit turn markers
> instead, which is why the next cell can verify them.


In [17]:
from unsloth.chat_templates import train_on_responses_only

trainer = train_on_responses_only(
    trainer,
    instruction_part = '<|turn>user\n',
    response_part    = '<|turn>model\n',
)
print('response masking applied')


Map (num_proc=6):   0%|          | 0/10895 [00:00<?, ? examples/s]

Map (num_proc=6):   0%|          | 0/256 [00:00<?, ? examples/s]

response masking applied


### Verify the mask — do not skip this

If the turn markers do not match the model's template, the mask covers nothing.
No error is raised. You train for hours against the wrong target and find out from
the metrics.

**What you should see:** the kept tokens start with the JSON answer. If the system
prompt or the user message appears there, stop and fix the markers — print
`tokenizer.apply_chat_template([...], tokenize=False)` to see the real ones.


In [18]:
ex     = trainer.train_dataset[0]
ids    = ex['input_ids']
labels = ex['labels']

kept    = [i for i, l in zip(ids, labels) if l != -100]
ignored = [i for i, l in zip(ids, labels) if l == -100]
share   = len(kept) / len(ids)

print(f'{len(ids)} tokens · {len(kept)} in the loss ({share*100:.0f}%)')
print('\n=== TOKENS THAT REACH THE LOSS ===')
print(_tok.decode(kept)[:600])
print('\n=== MASKED OUT ===')
print(_tok.decode(ignored)[:400])

assert 0.15 < share < 0.95, (
    'Suspicious mask ratio. Too low: the markers do not match. '
    'Too high: masking never applied.')
print('\nmask looks right')


391 tokens · 273 in the loss (70%)

=== TOKENS THAT REACH THE LOSS ===
{
  "issue_type": "Sub-task",
  "summary": "Wire the feature flag and rollout configuration",
  "description": "h2. Objective\nWire the feature flag and rollout configuration\n\nh2. Context\nPart of the story “Add recurring transfer to Transfers”. Single-person step, roughly 2 hours.\n\nh2. Steps to Reproduce\n# Implement the change in the module noted above\n# Cover it with a test at the level it belongs to\n# Open the pull request linked to the parent story\n\nh2. Done When\n* flag toggles the behaviour without a deploy",
  "priority": "Medium",
  "severity": null,
  "labels": [
    "payment

=== MASKED OUT ===
<|turn>system
You are a Jira issue writer for a product engineering team. Convert the user's input into structured issue fields as JSON. Keep summaries under 80 characters and in the imperative mood. Do not add a type prefix to the summary. Put every unstated detail into `assumptions` or `clarifying_questio

## 10 · Train

Checkpoints go to Drive, so a dropped session resumes from the last one — rerun
this cell and it picks up.

Watch the first 20 steps: multiply the step time by the total to get the real
duration. If it is longer than you can sit through, stop and set `EPOCHS = 1.0`.

> Unsloth's docs note that loss lands around 13–15 for the E2B/E4B variants
> (1–3 for 26B/31B). **Watch the trend, not the value** — and judge the run by the
> metrics in step 11, not by the loss.


In [19]:
resume = any(d.startswith('checkpoint-') for d in os.listdir(CKPT))
print('resuming' if resume else 'starting fresh')
stats = trainer.train(resume_from_checkpoint=resume)
print(stats.metrics)


The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.


starting fresh


==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 10,895 | Num Epochs = 2 | Total steps = 1,362
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 73,400,320 of 8,069,556,768 (0.91% trained)
Unsloth: Not an error, but Gemma4ForConditionalGeneration does not accept `num_items_in_batch`.
Using gradient accumulation will be very slightly less accurate.
Read more on gradient accumulation issues here: https://unsloth.ai/blog/gradient


Step,Training Loss,Validation Loss
200,0.089117,0.194344
400,0.046892,0.226326
600,0.039133,0.258828
800,0.031120,0.289718
1000,0.030973,0.294396
1200,0.027505,0.307500
1362,0.029105,0.307293


Filter:   0%|          | 0/256 [00:00<?, ? examples/s]

Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-400/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-600/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-800/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-1000/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-1200/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in /content/drive/MyDrive/issue-writer-ckpt/checkpoint-1362/tokenizer_config.json.


{'train_runtime': 5785.2459, 'train_samples_per_second': 3.766, 'train_steps_per_second': 0.235, 'total_flos': 6.631326906559055e+17, 'train_loss': 0.09748898171402634, 'epoch': 2.0}


## 11 · Measure again

Same examples, same metrics, from a test split whose content cores never appear in
training — this measures generalisation, not recall.

Expect the schema metrics to move most: `sections_ok` and `ac_shape_ok` from near
zero to the nineties. `type_acc` moves less, because the base model already
half-knows Story from Bug.

**If `sections_ok` does not move, the problem is the mask, not the training.**
Go back to step 9.


In [20]:
# Same function, so the same batching applies here.
AFTER = evaluate(n=25, label='FINE-TUNED')

print(f"\n{'metric':18}{'before':>9}{'after':>9}{'delta':>9}")
for k in sorted(set(BEFORE) | set(AFTER)):
    b, a = BEFORE.get(k, 0) * 100, AFTER.get(k, 0) * 100
    mark = '+' if a - b > 1 else ('-' if a - b < -1 else ' ')
    print(f'{k:18}{b:8.1f}%{a:8.1f}%{a-b:+8.1f} {mark}')

# The fine-tuned model emits EOS and stops, so this run is also faster than the
# baseline was: shorter outputs, same batch.

# At n=25 the confidence interval on a proportion is roughly +-12%, so small
# deltas are noise. Before making a decision, run evaluate(n=120).

# A metric that drops is worth chasing. no_hallucination falling means the
# dataset taught the model to invent something.


batch size 16
  16/25
  25/25

=== FINE-TUNED · 25 examples ===
  ac_count_ok        100.0%   (11/11)
  ac_shape_ok        100.0%   (11/11)
  has_all_fields     100.0%   (25/25)
  json_lenient       100.0%   (25/25)
  json_strict        100.0%   (25/25)
  no_hallucination   100.0%   (25/25)
  sections_ok        100.0%   (25/25)
  summary_ok         100.0%   (25/25)
  type_acc            96.0%   (24/25)

metric               before    after    delta
ac_count_ok           57.1%   100.0%   +42.9 +
ac_shape_ok            0.0%   100.0%  +100.0 +
has_all_fields         0.0%   100.0%  +100.0 +
json_lenient         100.0%   100.0%    +0.0  
json_strict            0.0%   100.0%  +100.0 +
no_hallucination     100.0%   100.0%    +0.0  
sections_ok            0.0%   100.0%  +100.0 +
summary_ok            72.0%   100.0%   +28.0 +
type_acc              28.0%    96.0%   +68.0 +


### Export the results

Prints the comparison as markdown so it can go straight into a README. Paste the
output into `BENCHMARK.md` in the repo and into the dataset card.


In [ ]:
from datetime import date

NAMES = {
    'json_strict':      'Bare JSON (no code fence)',
    'json_lenient':     'Parseable JSON',
    'has_all_fields':   'All required fields present',
    'sections_ok':      'Description has 3+ sections',
    'ac_shape_ok':      'Acceptance criteria well-formed',
    'ac_count_ok':      'Criteria count within 3-7',
    'summary_ok':       'Summary length and form',
    'type_acc':         'Issue type matches',
    'priority_acc':     'Priority matches',
    'no_hallucination': 'No invented version numbers',
}

lines = [
    '## Benchmark',
    '',
    f'Base: `{MODEL}` · LoRA r=32 · {EPOCHS:g} epochs on {len(ds["train"]):,} examples',
    f'Measured on 25 held-out test examples · greedy decoding · {date.today().isoformat()}',
    '',
    '| Check | Base model | Fine-tuned | Change |',
    '|---|---:|---:|---:|',
]
for k in sorted(set(BEFORE) | set(AFTER)):
    b, a = BEFORE.get(k, 0) * 100, AFTER.get(k, 0) * 100
    d = a - b
    arrow = 'up' if d > 1 else ('down' if d < -1 else '--')
    lines.append(f'| {NAMES.get(k, k)} | {b:.0f}% | {a:.0f}% | {arrow} {d:+.0f} |')

lines += [
    '',
    '> Measured with n=25, where the confidence interval on a proportion is roughly',
    '> +-12%. Differences under about 15 points are not distinguishable from noise.',
    '> The test split is built from content cores never seen in training.',
]
print('\n'.join(lines))


## 12 · Save and publish

Three artefacts, three jobs. You do not need all of them.

| Artefact | Size | For |
|---|---|---|
| LoRA adapter | ~150 MB | keeping the result; needs the base model to run |
| GGUF `q4_k_m` | ~2.5 GB | Ollama, llama.cpp, LM Studio |
| Merged 16-bit | ~16 GB | vLLM, transformers, TGI |

Start with the adapter and the GGUF. The merged weights are only worth the
upload if you are serving with vLLM.


In [21]:
from huggingface_hub import whoami

HF_USER = whoami()['name']
ADAPTER_REPO = f'{HF_USER}/issue-writer-gemma4-lora'
GGUF_REPO    = f'{HF_USER}/issue-writer-gemma4-gguf'
MERGED_REPO  = f'{HF_USER}/issue-writer-gemma4'

# Uploads are public by default. Set False to keep everything local.
PUSH_TO_HUB = True

OUT = os.path.abspath('./issue-writer-gemma4')
model.save_pretrained(OUT)
tokenizer.save_pretrained(OUT)
print('adapter saved locally to', OUT)

import shutil
if os.path.isdir('/content/drive/MyDrive'):
    shutil.copytree(OUT, '/content/drive/MyDrive/issue-writer-gemma4',
                    dirs_exist_ok=True)
    print('copied to Drive')

for name, repo in (('adapter', ADAPTER_REPO), ('gguf', GGUF_REPO),
                   ('merged', MERGED_REPO)):
    print(f'  {name:8} -> {repo}')


Unsloth: Restored added_tokens_decoder metadata in /content/issue-writer-gemma4/tokenizer_config.json.


adapter saved locally to /content/issue-writer-gemma4
copied to Drive
  adapter  -> fport/issue-writer-gemma4-lora
  gguf     -> fport/issue-writer-gemma4-gguf
  merged   -> fport/issue-writer-gemma4


### LoRA adapter

A minute or two. This is the artefact worth keeping regardless of how you serve:
everything else can be regenerated from it.


In [22]:
if PUSH_TO_HUB:
    model.push_to_hub(ADAPTER_REPO, token=os.environ['HF_TOKEN'])
    tokenizer.push_to_hub(ADAPTER_REPO, token=os.environ['HF_TOKEN'])
    print('https://huggingface.co/' + ADAPTER_REPO)
else:
    print('PUSH_TO_HUB is False — nothing uploaded')


README.md:   0%|          | 0.00/567 [00:00<?, ?B/s]

Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...adapter_model.safetensors:   0%|          |  558kB /  294MB            

Saved model to https://huggingface.co/fport/issue-writer-gemma4-lora


Unsloth: Restored added_tokens_decoder metadata in /tmp/tmpsoz4w_rf/tokenizer_config.json.


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...mpsoz4w_rf/tokenizer.json:  99%|#########9| 31.9MB / 32.2MB            

https://huggingface.co/fport/issue-writer-gemma4-lora


### GGUF for Ollama

Conversion takes 10–15 minutes and needs disk headroom; Colab has enough.

`q4_k_m` is the usual starting point — about a quarter of the size at a quality
loss most people cannot spot. **Do not assume that holds here.** This model fills
a strict schema, and quantisation degrades structured output before it degrades
prose. If the metrics drop after quantising, try `q5_k_m` or `q8_0` before
suspecting the training.

> Unsloth's export helpers move between releases. If the call is missing, the
> cell says so and falls back to a local export you can upload by hand.


In [23]:
QUANTS = ['q4_k_m']          # add 'q8_0' to compare quality against size

if PUSH_TO_HUB:
    try:
        model.push_to_hub_gguf(GGUF_REPO, tokenizer,
                               quantization_method=QUANTS,
                               token=os.environ['HF_TOKEN'])
        print('https://huggingface.co/' + GGUF_REPO)
    except AttributeError as e:
        print(f'push_to_hub_gguf unavailable in this build ({e})')
        print('falling back to a local export — upload it manually')
        model.save_pretrained_gguf('gguf', tokenizer,
                                   quantization_method=QUANTS[0])
        print('written to ./gguf')
else:
    print('PUSH_TO_HUB is False — skipping')


Unsloth: Converting model to GGUF format...
Unsloth: Merging model weights to 16-bit format...


Unsloth: Restored added_tokens_decoder metadata in /tmp/unsloth_gguf_bmn02kgf/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `/tmp/unsloth_gguf_bmn02kgf`: 100%|██████████| 1/1 [00:34<00:00, 34.91s/it]


Successfully copied all 1 files from cache to `/tmp/unsloth_gguf_bmn02kgf`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


...

Uploading gemma-4-E4B-it.BF16-mmproj.gguf...


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...4-E4B-it.BF16-mmproj.gguf:   3%|3         | 30.4MB /  992MB            

Uploading config.json...
Uploading Ollama Modelfile...
Unsloth: Successfully uploaded GGUF to https://huggingface.co/fport/issue-writer-gemma4-gguf
Unsloth: Cleaning up temporary files...
https://huggingface.co/fport/issue-writer-gemma4-gguf


### Merged 16-bit — only for vLLM

~16 GB to build and upload. Skip it unless you are serving with vLLM, and even
then consider serving the adapter directly with `--enable-lora`, which needs no
merge at all.


In [26]:
PUSH_MERGED = False          # flip on if you need vLLM-ready weights

if PUSH_TO_HUB and PUSH_MERGED:
    model.push_to_hub_merged(MERGED_REPO, tokenizer,
                             save_method='merged_16bit',
                             token=os.environ['HF_TOKEN'])
    print('https://huggingface.co/' + MERGED_REPO)
else:
    print('skipped — serve the adapter with vLLM --enable-lora instead')


skipped — serve the adapter with vLLM --enable-lora instead


## 13 · Run it locally

**Ollama** — pull the GGUF you just pushed, then write a `Modelfile` next to it:

```bash
huggingface-cli download $HF_USER/issue-writer-gemma4-gguf \
  --include "*Q4_K_M.gguf" --local-dir ./model
```


```dockerfile
FROM ./issue-writer-gemma4.Q4_K_M.gguf
PARAMETER temperature 0        # the output is a schema; sampling only breaks JSON
PARAMETER num_ctx 4096
PARAMETER num_predict 2048     # outputs run to ~1400 tokens at p95
SYSTEM """You are a senior agile delivery assistant..."""   # same prompt as training
```

```bash
ollama create issue-writer -f Modelfile
ollama run issue-writer "Turn this into an issue: cart empties when a guest logs in"
```

**vLLM** — serve the adapter without merging:

```bash
vllm serve unsloth/gemma-4-E4B-it \
  --enable-lora --lora-modules issue-writer=$HF_USER/issue-writer-gemma4-lora \
  --max-lora-rank 32 --port 8000
```

`--max-lora-rank` must be at least the `r` used in training, or the adapter is
rejected at load time.

An agent that drives this model, with a rule reviewer and a dashboard, lives in
[strands-issue-writer](https://github.com/fport/strands-issue-writer).


---
## What went wrong, and what each check is for

Every verification cell above exists because something failed without one.

**1. A 60-step demo is not training.** The first attempt used `max_steps=60`,
which showed the model 480 of 13,000 examples. Nothing was learned and nothing
errored. Train in epochs.

**2. Gemma 4's tokenizer is a Processor.** Being multimodal, its first positional
argument is `images`. Three separate calls broke on this: `tokenizer(text)`,
`apply_chat_template(...)` returning a string instead of tensors, and its
`tokenize=True` path demanding multimodal content lists. Use the inner text
tokenizer for token-level work.

**3. Double BOS.** The chat template emits `<bos>` and `SFTTrainer` adds another.
No error, just a worse model. Strip it, then assert it is gone.

**4. An unverified mask can cover nothing.** `train_on_responses_only` takes turn
markers as strings. If they do not match the template, masking silently does
nothing and you train on your own prompts. Print the tokens that reach the loss.

**5. 4-bit on a big GPU is slower, not faster.** On an A100 80GB the run was
producing 5 tokens/second with 69 GB idle. Quantisation solves memory; it costs
speed.

**6. Generation needs inference mode.** Unsloth leaves the model in training
configuration — KV cache off — and `generate()` then recomputes the whole sequence
per token. `for_inference()` turned a 2-hour measurement into 8 minutes.

**7. The inference prompt must match training exactly.** Rewording the system
prompt, even to remove a vendor name, moves the model off-distribution.

**8. Measure before, or you cannot claim after.** The base model scored 0% on
`sections_ok` and `has_all_fields`. Without those numbers, "it looks better" is
all you have.

**9. Check what your metric divides by.** `ac_shape_ok` counted issues that carry
no acceptance criteria as failures and reported 64% when the true value was 100%.

**10. Padding can eat most of your GPU.** Examples average ~640 tokens against a
2048 limit, so fixed-length batches spent 69% of every step on padding.
`group_by_length=True` brings that to ~1%. Measure your length distribution before
assuming the GPU is the bottleneck.

**11. Generating one prompt at a time wastes the card.** Measuring 25 examples
took half an hour on an A100 using 15 GB of 80. Batched generation with left
padding is 4–5× faster. Right padding here produces garbage — the model continues
from pad tokens.

**12. A dataset can teach hallucination.** After the first run every metric
improved except `no_hallucination`, which fell from 100% to 76%. The generator was
putting environment details and version numbers into bug bodies while the input
channels never mentioned them — 26% of examples taught exactly that. The model
learned it faithfully. **When a metric drops, suspect the data before the
training.**
